# Train DustyLM from Scratch

Train Dusty, a tiny ~8M parameter assistant that talks like a robot vacuum — from raw data to a chatting model.

**What you will do:**
1. Set up the environment (Colab, RunPod, or local)
2. Download the training datasets
3. Train the tokenizer
4. Prepare tokenized data
5. Pretrain the base model
6. Select the best base checkpoint
7. Fine-tune with SFT
8. Chat with your trained model

This notebook works on **Google Colab, RunPod, or any local machine**. The setup cell auto-detects your environment.

In [ ]:
# ── Environment Setup ────────────────────────────────────────────────
# Colab: clones the repo, installs dependencies, downloads datasets.
# Local / RunPod: assumes you are in the repo root with deps installed.

import sys
from pathlib import Path

if "google.colab" in sys.modules:
    !git clone https://github.com/mkhordoo/dusty-lm.git
    %cd dusty-lm
    !pip install -q uv && uv pip install --system -e .
    print("✓ Colab environment ready.")
elif not Path("dustylm").exists():
    raise RuntimeError(
        "Run this notebook from the repository root.\n"
        "  cd /path/to/dusty-lm && jupyter notebook"
    )
else:
    print("✓ Running from repo root. Dependencies assumed installed.")

In [ ]:
import torch

REPO_ROOT = Path.cwd()
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print("Repo root:", REPO_ROOT)
print("PyTorch:", torch.__version__)
print("Training device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Download the Datasets

Downloads TinyStories for pretraining and the Dusty SFT JSONL from [mkhordoo/dusty-chat](https://huggingface.co/datasets/mkhordoo/dusty-chat). If you already have `artifacts/datasets/dusty_pretrain.txt` and `artifacts/datasets/dusty_sft.jsonl`, skip this cell.

In [ ]:
!make download-datasets

## 2. Check the Training Files

Training needs Dusty text under `artifacts/datasets/`. This cell verifies the required files exist.

In [ ]:
required_sources = {
    "pretrain text": REPO_ROOT / "artifacts" / "datasets" / "dusty_pretrain.txt",
    "SFT chat JSONL": REPO_ROOT / "artifacts" / "datasets" / "dusty_sft.jsonl",
}

for label, path in required_sources.items():
    status = "✓ found" if path.exists() else "✗ missing"
    print(f"{label:16} {status:10} {path}")

## 3. Train the Tokenizer

Dusty uses a small 4096-token BPE vocabulary. This trains the tokenizer from the local Dusty text and writes `artifacts/tokenizers/dusty_tokenizer.json`.

In [ ]:
!make dusty-tokenizer

## 4. Prepare Tokenized Datasets

Turns raw text and chat JSONL into Hugging Face `datasets` saved on disk. The training loop reads these tokenized datasets directly.

In [ ]:
!make dusty-pretrain-data
!make dusty-sft-data

## 5. Pretrain Dusty

Pretraining teaches the base `dusty8m` model language patterns before SFT teaches it to answer user messages. `EPOCHS=1` is a smoke test; increase for a better model.

In [ ]:
!make dusty-pretrain EPOCHS=1 CHECKPOINT_EVERY_STEPS=50

## 6. Check the Pretrained Base Model

Before SFT, generate from the `dusty8m` base profile. This tells you whether the base checkpoint has learned stable text patterns.

In [ ]:
!make dusty-generate PROFILE=dusty8m PROMPT="deep in the forest, there lived a " TOP_P=0.9 TEMPERATURE=0.8

## 7. Compare Pretraining Step Checkpoints

If you saved step checkpoints, generate from a few candidates. Pick the checkpoint with the best stability and Dusty-like rhythm before SFT. Replace the step numbers with checkpoints that exist in your `artifacts/checkpoints/` folder.

In [ ]:
for step in [50, 100, 150]:
    print("=" * 80)
    print("CHECKPOINT_STEP:", step)
    !make dusty-generate PROFILE=dusty8m CHECKPOINT_STEP={step} PROMPT="deep in the forest, there lived a " TOP_P=0.9 TEMPERATURE=0.8

## 8. Promote the Best Base Checkpoint

SFT initializes from `artifacts/checkpoints/dusty8m.pt`. If a step checkpoint is better than the final one, copy it before running SFT.

In [ ]:
BEST_PRETRAIN_STEP = 100

!cp artifacts/checkpoints/dusty8m_step_{BEST_PRETRAIN_STEP}.pt artifacts/checkpoints/dusty8m.pt

# Verify the promoted checkpoint
!make dusty-generate PROFILE=dusty8m PROMPT="deep in the forest, there lived a " TOP_P=0.9 TEMPERATURE=0.8

## 9. Fine-Tune with SFT

The `sft_dusty8m` profile automatically initializes from `artifacts/checkpoints/dusty8m.pt`.

In [ ]:
!make dusty-sft-train EPOCHS=1 CHECKPOINT_EVERY_STEPS=50

## 10. Chat with Your Trained Model

In [ ]:
from dustylm.inference import Inference

engine = Inference(
    checkpoint_path=REPO_ROOT / "artifacts" / "checkpoints" / "dusty8m_sft.pt",
    tokenizer_path=REPO_ROOT / "artifacts" / "tokenizers" / "dusty_tokenizer.json",
    device=device,
)


def chat(prompt):
    response = engine.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=64,
        temperature=0.6,
        top_p=0.8,
    )
    return response["choices"][0]["message"]["content"].strip()


for prompt in ["who are you?", "what makes you happy", "where do you charge?"]:
    print(f"You> {prompt}")
    print(f"Dusty> {chat(prompt)}\n")

## Save Your Checkpoints

Hosted runtimes (Colab, RunPod) can disappear. Download checkpoints or copy them to persistent storage before shutting down.

In [ ]:
from pathlib import Path

for path in sorted(Path("artifacts/checkpoints").glob("*.pt")):
    print(path, f"{path.stat().st_size / 1024 / 1024:.1f} MB")

---

## Advanced: Generate Custom Training Data

The repo can generate synthetic data via OpenRouter. These steps may call an external model provider, so set your API key first. Once data exists, training is just the `make` commands above.

In [ ]:
# Optional: generate pretraining text and SFT chat examples.
# Uncomment these when your model/API credentials are configured.
# !make dusty-generate-pretrain
# !make dusty-generate-sft

### Changing Dusty's Persona

If you simply want to retrain Dusty, use the downloaded dataset from `make download-datasets` and keep the training flow unchanged.

If you want a different robot persona, create a new SFT dataset by editing the prompt inside `data_pipeline/generate_sft_dataset_with_fallback.py`. Change the instructions that define Dusty's personality, topics, and speaking style, then run `make dusty-generate-sft` to produce your new chat JSONL before training.

## Quick Reference

Run these in a terminal when you do not need the notebook:

```bash
make download-datasets
make dusty-tokenizer
make dusty-pretrain-data
make dusty-pretrain EPOCHS=1
make dusty-sft-data
make dusty-sft-train EPOCHS=1
make chat
```

That is the whole loop: download data → tokenizer → pretrain → SFT → chat.